# Notebook 05 — Results & Novelty Analysis (with Skeleton Views)

**Goal**: Explore all historical alpha results from DuckDB, including the new skeleton family trees:
- Pass rate over time / by direction
- Metric distributions (Sharpe, Fitness, Turnover, IC)
- Expression novelty distribution
- Failure reason analysis
- Expression clustering
- Bandit direction performance
- **NEW** Skeleton family tree: which skeleton spawned which qualified alphas
- **NEW** Per-skeleton output distribution
- **NEW** TrackBandit arm performance over time

**Requires**: Notebook 04 completed with at least a few simulation rounds.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.knowledge.skeleton_registry import SkeletonRegistry
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.search.bandit import DirectionBandit
from alpha_agent.search.novelty import NoveltyScorer

memory   = AlphaMemory()
registry = SkeletonRegistry()
store    = VectorStore()
bandit   = DirectionBandit(memory)

stats = memory.stats()
reg_stats = registry.stats()
print(f"Total alphas: {stats['total']} | Qualified: {stats['qualified']} | Pass rate: {stats['pass_rate']:.1%}")
print(f"Skeleton registry: {reg_stats['total']} total, {reg_stats['active']} active, {reg_stats['instances']} instances")

## 1. Load all data from DuckDB

In [ ]:
df = memory.to_dataframe()

# Parse metrics JSON
metrics_df = df['metrics_json'].apply(lambda x: json.loads(x) if x else {}).apply(pd.Series)
df = pd.concat([df.drop(columns=['metrics_json']), metrics_df], axis=1)

# Parse failure reasons
df['failure_reasons_list'] = df['failure_reasons'].apply(
    lambda x: json.loads(x) if x else []
)

print(f"Loaded {len(df)} rows")
display(df.head(5))

## 2. Pass rate over time

In [ ]:
if len(df) > 0:
    df['created_at'] = pd.to_datetime(df['created_at'])
    df = df.sort_values('created_at')
    df['cumulative_pass_rate'] = df['qualified'].expanding().mean()

    fig, ax = plt.subplots()
    ax.plot(df['created_at'], df['cumulative_pass_rate'], marker='o', ms=4)
    ax.set_title('Cumulative Pass Rate Over Time')
    ax.set_ylabel('Pass Rate')
    ax.set_xlabel('Time')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    plt.tight_layout()
    plt.show()
else:
    print("No data yet. Run Notebook 04 first.")

## 3. Metric distributions

In [ ]:
if len(df) > 0 and 'sharpe' in df.columns:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    metrics = ['sharpe', 'fitness', 'turnover', 'margin']
    thresholds = [1.5, 1.0, None, 0.02]

    for ax, metric, threshold in zip(axes, metrics, thresholds):
        values = df[metric].dropna()
        ax.hist(values, bins=20, edgecolor='white', alpha=0.8)
        if threshold is not None:
            ax.axvline(threshold, color='red', ls='--', label=f'threshold={threshold}')
            ax.legend(fontsize=8)
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')

    plt.suptitle('Alpha Metric Distributions', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No metric data available.")

## 4. Failure reason analysis

In [ ]:
if len(df) > 0:
    failed_df = df[~df['qualified']]
    all_reasons = [r for reasons in failed_df['failure_reasons_list'] for r in reasons]

    if all_reasons:
        # Categorize reasons
        reason_cats = {}
        for r in all_reasons:
            if 'sharpe' in r:    cat = 'Low Sharpe'
            elif 'fitness' in r:  cat = 'Low Fitness'
            elif 'turnover' in r: cat = 'Turnover OOB'
            elif 'ic_mean' in r:  cat = 'Low IC'
            elif 'check_fail' in r: cat = 'Check Fail'
            else:                   cat = 'Other'
            reason_cats[cat] = reason_cats.get(cat, 0) + 1

        fig, ax = plt.subplots(figsize=(8, 4))
        cats = list(reason_cats.keys())
        counts = list(reason_cats.values())
        ax.barh(cats, counts)
        ax.set_title('Failure Reason Breakdown')
        ax.set_xlabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print("No failure reasons recorded yet.")
else:
    print("No data available.")

## 5. Novelty distribution

In [ ]:
if len(df) > 1:
    novelty_scorer = NoveltyScorer(store, memory)
    sample = df['expression'].dropna().tolist()[:30]  # cap for speed
    scores = novelty_scorer.score_batch(sample)

    fig, ax = plt.subplots()
    ax.hist(scores, bins=15, edgecolor='white')
    ax.axvline(0.3, color='orange', ls='--', label='novelty threshold=0.3')
    ax.set_title('Expression Novelty Score Distribution')
    ax.set_xlabel('Novelty Score')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 alphas for novelty analysis.")

## 6. Expression clustering

In [ ]:
if len(df) >= 5:
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans

    sample_df = df[['expression', 'qualified', 'sharpe']].dropna().head(50)
    exprs = sample_df['expression'].tolist()
    embeddings = store.embed(exprs)
    emb_arr = np.array(embeddings)

    # PCA → 2D
    pca = PCA(n_components=2)
    coords = pca.fit_transform(emb_arr)

    # K-means clustering
    n_clusters = min(5, len(exprs))
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = km.fit_predict(emb_arr)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = cm.tab10(labels / n_clusters)
    qualified_mask = sample_df['qualified'].values
    markers = ['*' if q else 'o' for q in qualified_mask]
    sizes   = [200 if q else 60 for q in qualified_mask]

    for i, (x, y, c, m, s) in enumerate(zip(coords[:, 0], coords[:, 1], colors, markers, sizes)):
        ax.scatter(x, y, c=[c], marker=m, s=s, alpha=0.8, edgecolors='white', linewidths=0.5)

    ax.set_title('Expression Embedding Clusters (PCA 2D)\n★ = qualified  ● = not qualified')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 5 alphas for clustering.")

## 7. Bandit direction performance

In [ ]:
bandit_stats = bandit.stats_table()
df_bandit = pd.DataFrame(bandit_stats)

print("Bandit direction stats:")
display(df_bandit)

In [ ]:
# Qualified alphas summary table
qual_df = df[df['qualified'] == True][['expression', 'sharpe', 'fitness', 'turnover', 'margin', 'hypothesis']]
if len(qual_df) > 0:
    print("All qualified alphas:")
    display(qual_df.sort_values('sharpe', ascending=False))
else:
    print("No qualified alphas yet.")

## 8. Skeleton Family Tree

In [ ]:
# Per-skeleton stats table
skeletons = registry.all(include_archived=True)

if skeletons:
    skel_rows = []
    for sk in skeletons:
        skel_rows.append({
            "skeleton_id": sk.skeleton_id[:8],
            "template": sk.template_str[:60],
            "operators": ", ".join(sk.operators_used[:3]),
            "attempt_count": sk.attempt_count,
            "success_count": sk.success_count,
            "soft_success_count": sk.soft_success_count,
            "success_rate": f"{sk.success_rate:.0%}",
            "avg_sharpe": round(sk.avg_sharpe, 3),
            "archived": sk.archived,
        })
    df_skel = pd.DataFrame(skel_rows)
    print("Skeleton Registry — all skeletons:")
    display(df_skel)
else:
    print("No skeletons in registry yet. Run Notebook 04 with track_mode='hybrid' to populate.")

In [ ]:
# Skeleton family tree — link skeleton → alpha children
if len(df) > 0 and 'skeleton_id' in df.columns:
    skel_children = df[df['skeleton_id'].fillna('') != ''].groupby('skeleton_id').apply(
        lambda g: g[['id', 'expression', 'qualified', 'sharpe']].to_dict(orient='records')
    ).to_dict()

    print(f"Skeletons with alpha children: {len(skel_children)}\n")
    for sk_id, children in list(skel_children.items())[:5]:
        sk = registry.get(sk_id)
        print(f"SKELETON [{sk_id[:8]}] {sk.template_str[:70] if sk else '(not in registry)'}")
        for ch in children[:5]:
            icon = "✅" if ch['qualified'] else "❌"
            sharpe = f"sharpe={ch.get('sharpe', 0):.2f}" if ch.get('sharpe') else ""
            print(f"  {icon} {ch['expression'][:60]}  {sharpe}")
        print()
else:
    print("No skeleton_id column in alpha data or no skeleton-track alphas yet.")

In [ ]:
# Per-skeleton output distribution (bar chart of success rates)
if skeletons and any(sk.attempt_count > 0 for sk in skeletons):
    active_skel = [sk for sk in skeletons if sk.attempt_count > 0]
    labels = [sk.template_str[:30] + "..." for sk in active_skel]
    success_rates = [sk.success_rate for sk in active_skel]
    soft_rates = [sk.soft_rate for sk in active_skel]

    x = np.arange(len(active_skel))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(active_skel)*1.5), 5))
    bars1 = ax.bar(x - width/2, success_rates, width, label='Success Rate', color='steelblue')
    bars2 = ax.bar(x + width/2, soft_rates, width, label='Soft Rate', color='lightblue', alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Rate')
    ax.set_title('Per-Skeleton Success Rate')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No skeleton attempts recorded yet.")

## 9. TrackBandit Arm Curve

In [ ]:
# Track comparison: explorer vs skeleton pass rates (using alpha memory data)
if len(df) > 0 and 'track' in df.columns:
    track_df = df.groupby('track').agg(
        total=('id', 'count'),
        qualified=('qualified', 'sum'),
    ).reset_index()
    track_df['pass_rate'] = track_df['qualified'] / track_df['total']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart: total simulated per track
    axes[0].bar(track_df['track'], track_df['total'], color=['steelblue', 'seagreen'])
    axes[0].set_title('Total Simulated per Track')
    axes[0].set_ylabel('Count')

    # Bar chart: pass rates
    bars = axes[1].bar(track_df['track'], track_df['pass_rate'],
                        color=['steelblue', 'seagreen'])
    for bar, rate in zip(bars, track_df['pass_rate']):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.005,
                     f'{rate:.1%}', ha='center', fontsize=10)
    axes[1].set_title('Pass Rate per Track')
    axes[1].set_ylabel('Pass Rate')
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    plt.tight_layout()
    plt.show()
    display(track_df)
else:
    print("No track column data yet. Run Notebook 04 with dual-track mode.")

In [ ]:
# Optional: release DB/file handles before switching notebooks
import gc

for obj_name in ["memory", "registry", "store", "pi"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store", "pi"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

## ✅ Notebook 05 Complete

You now have full visibility into:
- How pass rates evolve with learning
- Which metric is the bottleneck
- Which research directions are most productive (bandit stats)
- Structural diversity of generated expressions (clustering)

Use these insights to tune the prompts, adjust qualification thresholds in `.env`,
or add new research papers to the knowledge base.